# Advanced Retrieval with LangChain

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

In [1]:
import os
import getpass

In [2]:
os.environ["TOGETHER_API_KEY"] = getpass.getpass("Enter your Together API Key:")

In [3]:
os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Loan Data once again - this time the strutured data available through the CSV!

### Data Preparation

We want to make sure all our documents have the relevant metadata for the various retrieval strategies we're going to be applying today.

In [20]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

path = "bills/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load_and_split(RecursiveCharacterTextSplitter(chunk_size=256, chunk_overlap=20))


Let's look at an example document to see if everything worked as expected!

In [21]:
docs[0]

Document(metadata={'producer': 'Skia/PDF m140 Google Docs Renderer', 'creator': '', 'creationdate': '2025-07-22T10:24:28+08:00', 'source': 'bills/HB02186.pdf', 'file_path': 'bills/HB02186.pdf', 'total_pages': 6, 'format': 'PDF 1.7', 'title': 'House Bill on Integrating AI in Basic Education Curriculum (BHPL).docx', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-07-22T10:24:28+08:00', 'trapped': '', 'encryption': 'Standard V5 R6 256-bit AES', 'modDate': "D:20250722102428+08'00'", 'creationDate': "D:20250722102428+08'00'", 'page': 0}, page_content='Republic of the Philippines \nHOUSE OF REPRESENTATIVES \nQuezon City, Metro Manila \nTWENTIETH CONGRESS \nFirst Regular Session \nHOUSE BILL NO. ______ \nIntroduced by \nRep. Robert Nazal \nAN ACT \nINSTITUTIONALIZING THE INTEGRATION OF ARTIFICIAL INTELLIGENCE')

## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "LoanComplaints".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [22]:
from langchain_together import ChatTogether

# choose from our 50+ models here: https://docs.together.ai/docs/inference-models
chat = ChatTogether(
    together_api_key=os.environ["TOGETHER_API_KEY"],
    model="allanctan_e665/openai/gpt-oss-20b-1eb68dc2",
)

# stream the response back from the model
for m in chat.stream("Tell me fun things to do in NYC"):
    print(m.content, end="", flush=True)


Here’s a handy “NYC Fun‑Guide” that covers the big sights and the little secrets, split by vibe and budget so you can pick what matches your mood.

---

## 1. Classic Must‑See
| Spot | Why it’s great | Quick tip |
|------|----------------|-----------|
| **Statue of Liberty / Ellis Island** | Iconic history, free ferry (for the Statue, just a ferry fee) | Buy tickets at “America’s 100th Birthday” (4‑5 p.m. Thursday‑Saturday). |
| **Top of The Rock / 30 Walmart** | 360° skyline, cheaper than Empire State | Go early or use the *Rock* $20 access pass; sky‑rail ride ($30) saves the line. |
| **Central Park** | Natural escape, walks, boat rides | Rent a bike (≈$20/day), or join a guided food‑walk at the park’s 5th‑Avenue entrance. |

---

## 2. Culture & Arts
| Spot | What to do | Budget |
|------|------------|--------|
| **Museum of Modern Art (MoMA)** | Hand‑picked contemporary works | $25, free on Fri 5‑8 p.m. |
| **The Cloisters** | Medieval art, lovely gardens | $18 (only on Tues, Fri, 

In [24]:
from langchain_community.vectorstores import Qdrant
from langchain_together import TogetherEmbeddings

embeddings = TogetherEmbeddings(model="BAAI/bge-large-en-v1.5", together_api_key=os.environ["TOGETHER_API_KEY"],)

vectorstore = Qdrant.from_documents(
    docs,
    embeddings,
    location=":memory:",
    collection_name="AI_Bills"
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [25]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [26]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [27]:
from langchain_together import ChatTogether

chat_model = ChatTogether(
    model="allanctan_e665/openai/gpt-oss-20b-1eb68dc2",
    together_api_key=os.environ["TOGETHER_API_KEY"],
)

response = chat_model.invoke("What are some fun things to do in NYC?")
print(response)

content='Here’s a quick, bite‑size guide to a mix of iconic, off‑the‑beaten‑path, and practical ways to enjoy New\u202fYork City:\n\n| What to Do | Why It’s Fun | Quick Tips |\n|------------|--------------|-------------|\n| **Explore a MoMA or Whitney** | World‑class art without the tourist throng if you go Wednesday evenings (free at 5‑10\u202fpm). | Check their “peek‑the‑future” preview nights—half‑price or free. |\n| **Walk the High Line** | A city park built on an old elevated freight rail line, with gardens, murals, and river views. | Arrive early on a weekday, or stroll at sunset on weekends. |\n| **Catch a Broadway or Off‑Broadway show** | The quintessential NYC experience. | Use the TKTS booth for discounted tickets; or buy tickets on the day of performance for last‑minute deals. |\n| **Ride the Staten Island Ferry** | Free cruise with great Statue of Liberty views. | Take it at sunrise for a lighter crowd and beautiful light. |\n| **Attend a live TV show taping** (e.g., *The T

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [28]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [34]:
markdown = naive_retrieval_chain.invoke({"question" : "What are the bills about?"})["response"].content

In [35]:
from IPython.display import Markdown, display

display(Markdown(markdown))

**House Bill HB 02186**  
- **Purpose:** To embed Artificial‑Intelligence (AI) education into the basic‑school curriculum.  
- **Goal:** Equip Filipino children and young adults with the knowledge, skills and ethical grounding needed to shape and use AI responsibly so that the nation can progress without compromising shared values.

**Senate Bill SBN 25 (also called SB 29 – “AI Regulation Act”)**  
- **Purpose:** To create a national framework that regulates how AI is developed and used in the Philippines.  
- **Key points:**  
  * Encourages innovation while providing safeguards.  
  * Requires transparency and meaningful human oversight.  
  * Addresses potential harms that could arise from AI, whether intentional or accidental, and supports Filipino technical ingenuity and progress.

**House Bill HB 07913**  
- **Context:** The document mainly deals with implementation provisions and appropriations for an AI‑related act (the specific subject line is not captured in the snippet).  
- **Function:** Provides rules and regulations, financial allocations, and administrative procedures needed to carry out the overarching AI legislation.

In summary, the bills collectively seek to **(1) integrate AI into the education system, (2) establish a regulatory framework for AI development and deployment, and (3) outline the operating and budgetary mechanisms needed to execute these policies.**

In [36]:
naive_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'I’m sorry, but I don’t have enough information on the specific handling of complaints to say whether any were not addressed in a timely manner.'

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [42]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(docs, k=10)

We'll construct the same chain - only changing the retriever.

In [43]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [44]:
from IPython.display import Markdown, display

markdown = bm25_retrieval_chain.invoke({"question" : "What are the bills about?"})["response"].content

display(Markdown(markdown))


The documents you’ve shown contain several different pieces of legislation that all relate to Artificial Intelligence in some way:

| Bill | Title / Reference | Main Focus |
|------|-------------------|------------|
| **HB 07913** | *House Bill 79‑13* | A broad AI‑governance bill that calls for transparency and accountability in the design and use of automated systems.  It specifically argues that creators (artists, gig‑workers, etc.) should have a say in how AI is deployed and that the state should enforce rules to prevent the misuse of AI or the unfair exploitation of labor and creative works. |
| **HB 02186** | *House Bill 21‑186 – “Integrating AI in Basic Education Curriculum”* | A bill that proposes adding AI concepts, skills, and tools to the Philippines’ basic‑education curriculum.  The idea is to prepare future learners for an AI‑driven economy and to improve national competitiveness. |
| **SB 29** | *Senate Bill 29 – AI Regulation Act* | A regulation bill that sets standards for the safe and ethical use of AI, requires developers to rectify harmful or unsafe outputs, and prohibits AI from being used for crime, fraud or other harmful activities.  It establishes oversight by an agency that can intervene if a system is unsafe. |
| **20250725 SBN 25** | *Senate Bill 25 (AI Regulation Act, 2025)* | A companion or updated version of SB 29 that again seeks to regulate AI through safety, accountability, and oversight, and reinforces the need for developers to correct hallucinations, protect public safety, and avoid harmful uses. |

**In short:**  
- HB 07913 is an AI‑regulation bill that emphasizes transparency, accountability, and worker/creator protection.  
- HB 02186 is an educational bill that wants AI concepts embedded in the basic curriculum.  
- SB 29 and SBN 25 are AI‑regulation acts focused on safety, prevention of misuse, and developer responsibility.

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

#### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.